In [28]:
import pandas as pd
import numpy as np

In [29]:
data = pd.read_csv('data_all_num_bank_cleaned.csv')

In [30]:
index_loans = data.columns.to_list().index('Type_of_Loan')
data_1 = data.iloc[:, :index_loans].copy()
data_2 = data.iloc[:, index_loans+1:].copy()
data_loans = data.iloc[:, index_loans].copy()
data_loans = pd.DataFrame(data_loans)
data_loans['Type_of_Loan'] = data_loans['Type_of_Loan'].str.replace(' and ', ' ')

In [31]:
data_loans

,Type_of_Loan
0,"Mortgage Loan, Mortgage Loan, Not Specified"
1,"Mortgage Loan, Mortgage Loan, Not Specified"
2,"Mortgage Loan, Mortgage Loan, Not Specified"
3,"Mortgage Loan, Mortgage Loan, Not Specified"
4,"Mortgage Loan, Mortgage Loan, Not Specified"
...,...
92133,"Debt Consolidation Loan, Home Equity Loan, Cre..."
92134,"Debt Consolidation Loan, Home Equity Loan, Cre..."
92135,"Debt Consolidation Loan, Home Equity Loan, Cre..."
92136,"Debt Consolidation Loan, Home Equity Loan, Cre..."


In [32]:
def function_type_loan(x):
    try:
        loan_list = x.split(',')
        loan_count = {}
        for loan in loan_list:
            loan = loan.strip()
            if loan in loan_count:
                loan_count[loan] += 1
            else:
                loan_count[loan] = 1
        
        return loan_count
    except:
        return {}

data_loans['Loan_Counts'] = data_loans['Type_of_Loan'].apply(function_type_loan)
# Expand the dictionaries into separate columns
loan_counts_df = pd.json_normalize(data_loans['Loan_Counts'])

data_loans_expanded = pd.concat([data_1, loan_counts_df, data_2], axis=1)


In [33]:
data_loans_expanded.head()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Mortgage Loan,Not Specified,Personal Loan,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,38,86142.96,7274.58,0,1,7,3,2.0,1.0,NaN,...,Good,0.23,24.095879,20 Years and 11 Months,No,115.892037,536.182744,LowspentLargevaluepayments,345.383219,Good
1,38,86142.96,7274.58,0,1,7,3,2.0,1.0,NaN,...,Good,0.23,22.760102,21 Years and 0 Months,No,17656.000000,635.460243,NaN,266.105720,Good
2,38,86142.96,7274.58,0,1,7,3,2.0,1.0,NaN,...,Good,0.23,40.704485,21 Years and 2 Months,No,115.892037,220.132610,HighspentMediumvaluepayments,641.433353,Good
3,38,86142.96,7274.58,0,1,7,3,2.0,1.0,NaN,...,Good,0.23,37.451859,21 Years and 3 Months,NM,115.892037,447.321534,LowspentMediumvaluepayments,444.244429,Standard
4,38,86142.96,7274.58,0,1,7,3,2.0,1.0,NaN,...,Good,0.23,31.838221,21 Years and 4 Months,No,115.892037,675.923863,LowspentMediumvaluepayments,215.642100,Good


In [34]:
value_cols = ["Age", "Num_Credit_Card", "Interest_Rate", "Num_of_Loan", "Delay_from_due_date", "Num_of_Delayed_Payment", 
              "Num_Credit_Inquiries", "Changed_Credit_Limit", "Outstanding_Debt", "Credit_Utilization_Ratio", "Total_EMI_per_month", "Amount_invested_monthly"]

for col in value_cols:
    data_loans_expanded[col] = np.where(data_loans_expanded[col] < 0, 0, data_loans_expanded[col])

    upper_limit = data_loans_expanded[col].quantile(0.99)
    lower_limit = data_loans_expanded[col].quantile(0.01)
    data_loans_expanded[col] = np.where(data_loans_expanded[col] > upper_limit, upper_limit, data_loans_expanded[col])
    data_loans_expanded[col] = np.where(data_loans_expanded[col] < lower_limit, lower_limit, data_loans_expanded[col])

    data_loans_expanded[col] = data_loans_expanded[col].fillna(data_loans_expanded[col].mean())


In [35]:
def fun_credit_history_age(x):
    try:
        list_age = x.split(',')
        if len(list_age) == 2:
            years = int(list_age[0].split(" ")[0].strip())
            months = int(list_age[1].split(" ")[0].strip())
            return (years*12) + (months)
        elif len(list_age) == 1:
            months = int(list_age[0].split(" ")[0].strip())
            return months
    except:
        return 0
    
data_loans_expanded['Credit_History_Age'] = data_loans_expanded['Credit_History_Age'].apply(fun_credit_history_age)

In [37]:
data_loans_expanded.to_csv('data_all_num_bank_cleaned_expanded.csv', index=False)